# AIDA 1141 Final Project – 06 Single Test Evaluation

**Data version 2** (`shared/synthetic-data/`), using the employee split fixed in notebook 01, the model and thresholds saved in notebook 02, and the review policy frozen in notebook 04. Nothing is refitted, no threshold changes, and the policy choice is not reopened.

**Question:** How does the frozen early-warning policy perform on employees it has never seen, and what does that mean for the coordinator's review work?

**What is frozen.** Policy (a): flag an early-warning assessment when the task's start date is **8 or fewer business days** away; notebook 02 chose 8 on training data, and notebook 04 chose this policy on validation. The comparison model (logistic regression) and its full-precision threshold, the "never at risk" reference, the repair-check definition (notebook 05) and the test split are also fixed. The notebook first checks every input against the fingerprints saved with the frozen policy and stops if any has changed.

**What this notebook reports**, separately from validation (validation appears only as context):

- **Population:** test employees, tasks and early-warning assessments by checkpoint; overdue assessments (already past the deadline, reported as the separate urgent list); assessments whose outcome was unknown at the extract (excluded and counted). Task-level results use the tasks in the scored early-warning population as their denominator, not every task with a test assessment, and the employees represented in scoring are reported separately from all test employees.
- **Assessment level:** recall (the primary measure), precision, flags and the share flagged, and a coordinator view per 100 assessments.
- **Task level:** review volume (unique tasks flagged), at-risk tasks **flagged at least once**, and at-risk tasks **flagged before the start-date deadline** — the timing of a flag is checked from its timestamp, so eventual coverage and timely warning are reported separately.
- **Missed warnings:** every at-risk assessment the policy did not flag, with whether a later assessment of the same task was flagged and how many business days remained when it was.
- **Groups:** results by employee type, department, task type, intake channel and checkpoint, labelled with notebook 04's three categories, read from the frozen policy: **limited recall support** (fewer than 30 at-risk assessments); **documented concern** (recall at least 10 percentage points below the policy's own overall recall, or an at-risk task with no early-warning flag at any checkpoint); **serious concern** (at least 30 at-risk assessments and recall below 80%).
- **Comparison methods:** the model and "never at risk" on the same measures, with ranking measures; they are reported, not chosen.
- **Cleaning repairs:** notebook 05's procedure applied unchanged, including its audit of the rule's inputs.

A missing measure is reported as missing (for example precision when nothing is flagged), never as zero. The results are saved in `test_results.json` with the input fingerprints (including those of the repair-check definition, the generator and the frozen policy file), the thresholds and the population counts; reruns compare the results with the saved record and stop if they differ. A poor test result is reported and carried into the decision brief; it does not change the policy here.

All records are synthetic: the rates describe the generator's assumptions, not real employees or real HR performance.

In [1]:
import hashlib
import importlib.util
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

DATA_VERSION = "2"
SD = Path("../../../shared/synthetic-data")          # data version 2
OUTPUT_DIR = Path("../outputs")

read = lambda path: pd.read_csv(path, dtype=str, keep_default_na=False)
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
settings = json.loads((OUTPUT_DIR / "model_settings.json").read_text())          # notebook 02
policy = json.loads((OUTPUT_DIR / "review_policy.json").read_text())             # notebook 04 (frozen)
repair_definition = json.loads((OUTPUT_DIR / "repair_check_definition.json").read_text())   # notebook 05 (protected)
requests = read(SD / "cleaning/output/requests_cleaned.csv")
raw_requests = read(SD / "requests/data/requests.csv")
assessments = read(SD / "requests/data/assessments.csv")
events = read(SD / "requests/data/status_events.csv")
employee_split = read(OUTPUT_DIR / "employee_split.csv")
feature_list = read(OUTPUT_DIR / "feature_list.csv")
split_summary_01 = pd.read_csv(OUTPUT_DIR / "split_summary.csv", index_col="split")
cleaning_info = json.loads((SD / "cleaning/output/requests_cleaned_info.json").read_text())

# Every input must be the one the frozen policy was chosen with
FILES = {"requests_cleaned.csv": SD / "cleaning/output/requests_cleaned.csv",
         "assessments.csv": SD / "requests/data/assessments.csv",
         "employee_split.csv": OUTPUT_DIR / "employee_split.csv", "feature_list.csv": OUTPUT_DIR / "feature_list.csv",
         "model_settings.json": OUTPUT_DIR / "model_settings.json", "model.joblib": OUTPUT_DIR / "model.joblib",
         "validation_predictions.csv": OUTPUT_DIR / "validation_predictions.csv"}
FINGERPRINTS = {n: sha(p) for n, p in FILES.items()}
assert FINGERPRINTS == policy["fingerprints_sha256"], "An input differs from the fingerprints saved with the frozen policy (notebook 04)"
assert policy["chosen_policy"] == "a" and not policy["model_used"], "The frozen policy is not the days rule"
assert sha(SD / "requests/data/requests.csv") == cleaning_info["source_requests_sha256"] \
    and sha(SD / "requests/data/status_events.csv") == cleaning_info["source_events_sha256"], \
    "The raw export or event log differs from the ones the cleaned file was made from"
assert repair_definition["applies_to"].startswith("validation here; notebooks 06-07 apply the same rule unchanged")
# Further fingerprints for the test-run record. No earlier artifact of this project records hashes of these three files,
# so there is no expected value to compare with here; reruns compare them with the saved test record instead.
RUN_FINGERPRINTS = {**FINGERPRINTS, "review_policy.json": sha(OUTPUT_DIR / "review_policy.json"),
                    "repair_check_definition.json": sha(OUTPUT_DIR / "repair_check_definition.json"),
                    "generate_requests.py": sha(SD / "requests/generate_requests.py")}
assert sklearn.__version__ == settings["package_versions"]["scikit-learn"], "scikit-learn differs from the version the model was saved with"

model = joblib.load(OUTPUT_DIR / "model.joblib")
FEATURES = settings["features"]
NUMERIC = feature_list.loc[feature_list["type"] == "numeric", "feature"].tolist()
DAYS_THRESHOLD = policy["threshold"]
MODEL_THRESHOLD = settings["model_threshold"]
assert DAYS_THRESHOLD == settings["days_threshold"] and policy["flag_rule"] == f"business_days_until_start <= {DAYS_THRESHOLD}"
# Notebook 04's group rules, read from the frozen policy so test and validation use the same categories
criteria = policy["decision_criteria"]
GROUP_RULES = {"limited_support_below_at_risk": criteria["limited_recall_support_below_at_risk"],
               "documented_concern_gap_pp": criteria["documented_concern_gap_pp_vs_own_overall"],
               "serious_concern_recall_below_pct": criteria["serious_concern_recall_below_pct"]}
assert GROUP_RULES == {"limited_support_below_at_risk": 30, "documented_concern_gap_pp": -10, "serious_concern_recall_below_pct": 80}

# Business-day calendar from the data version 2 generator
spec = importlib.util.spec_from_file_location("generator", SD / "requests/generate_requests.py")
generator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generator)

# Notebook 02's input table, rebuilt for every split; the model must still reproduce its validation scores
data = (assessments
        .merge(requests[["request_id", "employee_id", "task_type", "hiring_department", "employee_type", "intake_channel",
                         "info_issue_at_intake"]].rename(columns={"employee_id": "request_employee_id"}),
               on="request_id", how="left", validate="many_to_one")
        .merge(employee_split[["employee_id", "split"]], on="employee_id", how="left", validate="many_to_one"))
data["intake_channel"] = data["intake_channel"].replace("", "unknown")   # missing at source (notebook 01)
data[NUMERIC] = data[NUMERIC].astype(int)
data["assessed_at"] = pd.to_datetime(data["assessed_at"], format="%Y-%m-%dT%H:%M")
validation_02 = pd.read_csv(OUTPUT_DIR / "validation_predictions.csv")
val_rows = data.set_index("assessment_id").loc[validation_02["assessment_id"]]
MODEL_REPRODUCED = bool(np.allclose(model.predict_proba(val_rows[FEATURES])[:, 1], validation_02["risk_probability"], rtol=0, atol=1e-12))
assert MODEL_REPRODUCED, "The saved model does not reproduce notebook 02's validation scores"

print(f"Data version {DATA_VERSION}; scikit-learn {sklearn.__version__}; inputs match the frozen policy's fingerprints")
print(f"Frozen policy (a): {policy['flag_rule']}; comparison model threshold {MODEL_THRESHOLD:.4f} (full precision saved)")
print(f"The saved model reproduces notebook 02's {len(validation_02):,} validation scores")
print(f"Group rules from the frozen policy: {GROUP_RULES}")

Data version 2; scikit-learn 1.9.1; inputs match the frozen policy's fingerprints
Frozen policy (a): business_days_until_start <= 8; comparison model threshold 0.1338 (full precision saved)
The saved model reproduces notebook 02's 1,033 validation scores
Group rules from the frozen policy: {'limited_support_below_at_risk': 30, 'documented_concern_gap_pp': -10, 'serious_concern_recall_below_pct': 80}


## A. The test population

The test split holds the employees notebook 01 set aside; none of them was used to fit the model, choose a threshold or choose the policy. The cell below rebuilds the test assessments with notebook 01's rules and checks the counts against notebook 01's split summary before any prediction or result is calculated:

- **Early-warning assessments** (outcome known, made before the task's deadline) are the scored population.
- **Overdue assessments** (made at or after the deadline) are already late; under the policy they go to the separate urgent-review list and are reported separately.
- **Unknown-outcome assessments** (task unfinished at the extract with its deadline still ahead) are excluded from scoring and counted.

The population table gives assessments, tasks and employees for each group and checkpoint, and the share of early-warning assessments labelled at risk, which notebook 01 already reported for the split.

In [2]:
test = data[data["split"] == "test"].copy()
early = test[(test["at_risk"] != "") & (test["past_deadline_at_assessment"] == "False")].copy()
early["y"] = (early["at_risk"] == "True").astype(int)
overdue = test[(test["at_risk"] != "") & (test["past_deadline_at_assessment"] == "True")].copy()
unknown = test[test["at_risk"] == ""].copy()

other_ids = set(data.loc[data["split"] != "test", "employee_id"])
s01 = split_summary_01.loc["test"]
POPULATION_OK = bool(
    len(early) + len(overdue) + len(unknown) == len(test)
    and not set(test["employee_id"]) & other_ids
    and (test["employee_id"] == test["request_employee_id"]).all()
    and test["employee_id"].nunique() == s01["employees"]
    and len(early) == s01["early_warning_assessments"] and int(early["y"].sum()) == s01["early_warning_at_risk"]
    and len(overdue) == s01["overdue_assessments"] and len(unknown) == s01["unknown_outcome_assessments"]
    and early[FEATURES].notna().all().all() and early[FEATURES].ne("").all().all()
    and early["assessment_id"].is_unique)
assert POPULATION_OK, "The rebuilt test population does not match notebook 01"


def population_rows(frame, group, labelled):
    rows = []
    for cp, g in [("all", frame)] + sorted(frame.groupby("checkpoint")):
        rows.append({"group": group, "checkpoint": cp, "assessments": len(g), "tasks": g["request_id"].nunique(),
                     "employees": g["employee_id"].nunique(),
                     "at_risk": int((g["at_risk"] == "True").sum()) if labelled else pd.NA})
    return rows


population = pd.DataFrame(population_rows(early, "early warning (scored)", True) + population_rows(overdue, "overdue (urgent list)", True)
                          + population_rows(unknown, "unknown outcome (excluded)", False))
population["at_risk"] = population["at_risk"].astype("Int64")
population["at_risk_%"] = (100 * population["at_risk"].astype(float) / population["assessments"]).where(
    population["group"] == "early warning (scored)")
population.insert(0, "data_version", DATA_VERSION)
population.to_csv(OUTPUT_DIR / "test_population.csv", index=False)
SCORED_TASKS, SCORED_EMPLOYEES = early["request_id"].nunique(), early["employee_id"].nunique()
print(f"Test split: {test['employee_id'].nunique()} employees and {test['request_id'].nunique()} tasks with any assessment; "
      f"counts match notebook 01")
print(f"Scored early-warning population: {len(early):,} assessments ({int(early['y'].sum())} at risk) on {SCORED_TASKS} tasks "
      f"from {SCORED_EMPLOYEES} employees (the task-level denominator); also {len(overdue)} overdue and {len(unknown)} unknown-outcome assessments")
display(population.drop(columns="data_version").round(1))

Test split: 215 employees and 877 tasks with any assessment; counts match notebook 01
Scored early-warning population: 1,056 assessments (192 at risk) on 829 tasks from 204 employees (the task-level denominator); also 41 overdue and 7 unknown-outcome assessments


,group,checkpoint,assessments,tasks,employees,at_risk,at_risk_%
0,early warning (scored),all,1056,829,204,192,18.2
1,early warning (scored),checkpoint_1,829,829,204,146,17.6
2,early warning (scored),checkpoint_2,227,227,103,46,20.3
3,overdue (urgent list),all,41,41,12,41,NaN
4,overdue (urgent list),checkpoint_1,41,41,12,41,NaN
5,unknown outcome (excluded),all,7,7,3,<NA>,NaN
6,unknown outcome (excluded),checkpoint_1,7,7,3,<NA>,NaN


## B. Results on the test set

Each method is applied once to the 1,056 scored early-warning assessments: the frozen policy (a) (business days until start ≤ 8), the comparison model at its saved full-precision threshold, and "never at risk". Measures are notebook 02's, with missing values reported as missing:

- **Assessment level:** confusion counts, recall (primary), precision, flags and the share flagged, and the coordinator view per 100 assessments.
- **Task level** (denominator: the 829 scored tasks): unique tasks flagged (review volume), at-risk tasks flagged at least once, at-risk tasks flagged before the start-date deadline, at-risk tasks never flagged, and the business days remaining at each at-risk task's **first** flag. "Before the start-date deadline" is checked from timestamps: the flag's assessment time against the task's deadline, 08:00 on its confirmed start date. Because early-warning assessments are by definition made before the deadline, the two counts should agree; the check confirms it rather than assuming it. How early a task was flagged is what the days remaining show.
- **Ranking:** ROC AUC and average precision for the model's probability and for days until start (fewer days = higher risk), only when both outcomes are present.
- **Overdue:** how many of the 41 overdue assessments each method would flag; under the policy all of them go to the urgent-review list anyway.

Validation results (notebook 02) are shown beside the test results for context only. The test results are recorded in `test_results.json` with the run's fingerprints, thresholds and population; a rerun compares its results with that record and stops if they differ.

In [3]:
early["risk_probability"] = model.predict_proba(early[FEATURES])[:, 1]
early["never_at_risk"] = False
early["days_rule"] = early.eval(policy["flag_rule"])                       # the frozen rule's saved expression
early["logistic_regression"] = early["risk_probability"] >= MODEL_THRESHOLD
METHODS = {"never_at_risk": "Never at risk (reference)", "days_rule": f"Policy (a): days until start <= {DAYS_THRESHOLD}",
           "logistic_regression": "Logistic regression (comparison)"}
# "Before the start-date deadline" is checked from timestamps: the task's deadline is 08:00 on its confirmed start date
start_date = early["request_id"].map(requests.set_index("request_id")["confirmed_start_date"])
early["deadline_at"] = pd.to_datetime(start_date, format="%Y-%m-%d") + pd.Timedelta(hours=8)
assert early["deadline_at"].notna().all(), "Every scored task needs a confirmed start date"
BEFORE_START = bool((early["assessed_at"] < early["deadline_at"]).all())
assert BEFORE_START, "A scored early-warning assessment is not before its task's deadline"


def evaluate(rows, flag):
    """Notebook 02's measures, plus timing at each at-risk task's first flag; missing measures stay missing."""
    y, p = rows["y"] == 1, rows[flag].astype(bool)
    tp, fp, fn, tn = int((p & y).sum()), int((p & ~y).sum()), int((~p & y).sum()), int((~p & ~y).sum())
    tasks = rows.assign(flag=p).groupby("request_id").agg(at_risk=("y", "max"), flagged=("flag", "any"))
    first = rows[p & y].sort_values("assessed_at").groupby("request_id").first()      # first flag of each at-risk task
    before_start = int((first["assessed_at"] < first["deadline_at"]).sum())            # checked from timestamps
    at_risk_tasks = int(tasks["at_risk"].sum())
    return {"assessments": len(rows), "at_risk": int(y.sum()), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "recall": tp / (tp + fn) if tp + fn else np.nan, "precision": tp / (tp + fp) if tp + fp else np.nan,
            "flagged_assessments": int(p.sum()), "flagged_%": 100 * p.mean(),
            "unique_tasks_flagged": int(tasks["flagged"].sum()), "scored_tasks": len(tasks), "at_risk_tasks": at_risk_tasks,
            "at_risk_tasks_flagged_at_least_once": int((tasks["at_risk"].astype(bool) & tasks["flagged"]).sum()),
            "at_risk_tasks_flagged_before_start_date": before_start,
            "at_risk_tasks_never_flagged": int((tasks["at_risk"].astype(bool) & ~tasks["flagged"]).sum()),
            "median_business_days_left_at_first_flag": float(first["business_days_until_start"].median()) if len(first) else np.nan,
            "at_risk_tasks_first_flagged_with_5_or_fewer_days_left": int((first["business_days_until_start"] <= 5).sum())}


results = pd.DataFrame([evaluate(early, m) for m in METHODS], index=list(METHODS.values()))
results.index.name = "method"
validation_context = pd.read_csv(OUTPUT_DIR / "validation_comparison.csv", index_col="method")
both_classes = early["y"].nunique() == 2
ranking = pd.DataFrame({
    "ROC AUC": [roc_auc_score(early["y"], early["risk_probability"]) if both_classes else np.nan,
                roc_auc_score(early["y"], -early["business_days_until_start"]) if both_classes else np.nan],
    "average precision": [average_precision_score(early["y"], early["risk_probability"]) if both_classes else np.nan,
                          average_precision_score(early["y"], -early["business_days_until_start"]) if both_classes else np.nan],
}, index=["Logistic regression (probability)", "Days until start (fewer days = higher risk)"])
overdue_flags = pd.Series({
    "overdue assessments (urgent-review list under the policy)": len(overdue),
    "would also be flagged by the days rule": int(overdue.eval(policy["flag_rule"]).sum()),
    "would also be flagged by the model": int((model.predict_proba(overdue[FEATURES])[:, 1] >= MODEL_THRESHOLD).sum()) if len(overdue) else 0,
}, name="test")

rule = results.loc[METHODS["days_rule"]]
n = len(early)
coordinator = pd.Series({"flagged": 100 * rule["flagged_assessments"] / n, "flagged and labelled at risk": 100 * rule["TP"] / n,
                         "flagged and labelled not at risk": 100 * rule["FP"] / n,
                         "at risk and not flagged (missed at that assessment)": 100 * rule["FN"] / n}, name="per 100 test assessments")

results.insert(0, "data_version", DATA_VERSION)
results.to_csv(OUTPUT_DIR / "test_comparison.csv")
ranking.to_csv(OUTPUT_DIR / "test_ranking.csv")
overdue_flags.to_frame().to_csv(OUTPUT_DIR / "test_overdue.csv")
early[["assessment_id", "request_id", "employee_id", "checkpoint", "assessed_at", "task_type", "employee_type", "y",
       "business_days_until_start", "risk_probability", "days_rule", "logistic_regression"]].rename(
    columns={"y": "at_risk"}).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

whole = lambda x: f"{int(x):,}"
print(f"Policy (a) on the test set: recall {100 * rule['recall']:.1f}% ({whole(rule['TP'])} of {whole(rule['at_risk'])}), precision "
      f"{100 * rule['precision']:.1f}%, {whole(rule['flagged_assessments'])} flags ({rule['flagged_%']:.1f}% of assessments) on "
      f"{whole(rule['unique_tasks_flagged'])} of {whole(rule['scored_tasks'])} scored tasks")
print(f"At-risk tasks: {whole(rule['at_risk_tasks_flagged_at_least_once'])} of {whole(rule['at_risk_tasks'])} flagged at least once; "
      f"{whole(rule['at_risk_tasks_flagged_before_start_date'])} first flagged before the 08:00 deadline on the start date (checked from "
      f"timestamps); median {rule['median_business_days_left_at_first_flag']:g} business days left at the first flag")
display(results.drop(columns="data_version").round(3).T)
print("Validation (notebook 02), for context only:")
display(validation_context.drop(columns="data_version").round(3).T)
display(ranking.round(3))
display(coordinator.round(1).to_frame())
display(overdue_flags.to_frame())

Policy (a) on the test set: recall 90.6% (174 of 192), precision 34.6%, 503 flags (47.6% of assessments) on 373 of 829 scored tasks
At-risk tasks: 146 of 146 flagged at least once; 146 first flagged before the 08:00 deadline on the start date (checked from timestamps); median 4 business days left at the first flag


method,Never at risk (reference),Policy (a): days until start <= 8,Logistic regression (comparison)
assessments,1056.0,1056.000,1056.000
at_risk,192.0,192.000,192.000
TP,0.0,174.000,171.000
FP,0.0,329.000,330.000
FN,192.0,18.000,21.000
TN,864.0,535.000,534.000
recall,0.0,0.906,0.891
precision,NaN,0.346,0.341
flagged_assessments,0.0,503.000,501.000
flagged_%,0.0,47.633,47.443


Validation (notebook 02), for context only:


method,Never at risk (reference),Days until start <= 8,Logistic regression
assessments,1033.0,1033.000,1033.000
at_risk,194.0,194.000,194.000
TP,0.0,177.000,178.000
FP,0.0,309.000,316.000
FN,194.0,17.000,16.000
TN,839.0,530.000,523.000
recall,0.0,0.912,0.918
precision,NaN,0.364,0.360
flagged_assessments,0.0,486.000,494.000
flagged_%,0.0,47.047,47.822


,ROC AUC,average precision
Logistic regression (probability),0.871,0.676
Days until start (fewer days = higher risk),0.860,0.602


,per 100 test assessments
flagged,47.6
flagged and labelled at risk,16.5
flagged and labelled not at risk,31.2
at risk and not flagged (missed at that assessment),1.7


,test
overdue assessments (urgent-review list under the policy),41
would also be flagged by the days rule,41
would also be flagged by the model,41


### Missed warnings, and what happened next

Every at-risk assessment the policy did not flag is listed with its task, checkpoint, days until start and status. For each one, the other assessments of the same task are checked **by timestamp**: whether a later assessment was flagged, when, and with how many business days left, and whether an earlier one had been. Two questions are kept apart: whether the miss was **followed up** by a later flag, and whether the task was **ever warned** at any of its scored assessments. A miss followed by a later flag still left the task without a warning at that earlier point.

In [4]:
flag = early["days_rule"].astype(bool)
missed = early[~flag & (early["y"] == 1)].sort_values("assessed_at")
later_rows = []
for m in missed.itertuples():
    later = early[(early["request_id"] == m.request_id) & (early["assessed_at"] > m.assessed_at)].sort_values("assessed_at")
    later_flagged = later[later["days_rule"].astype(bool)]
    first_later = later_flagged.iloc[0] if len(later_flagged) else None
    later_rows.append({"request_id": m.request_id, "assessment_id": m.assessment_id, "checkpoint": m.checkpoint,
                       "assessed_at": m.assessed_at, "business_days_until_start": m.business_days_until_start,
                       "current_status": m.current_status, "task_type": m.task_type,
                       "later_assessments": len(later), "flagged_later": first_later is not None,
                       "later_flag_assessment_id": first_later["assessment_id"] if first_later is not None else "",
                       "later_flag_at": first_later["assessed_at"] if first_later is not None else pd.NaT,
                       "business_days_left_at_later_flag": first_later["business_days_until_start"] if first_later is not None else pd.NA})
missed_table = pd.DataFrame(later_rows, columns=["request_id", "assessment_id", "checkpoint", "assessed_at", "business_days_until_start",
                                                  "current_status", "task_type", "later_assessments", "flagged_later",
                                                  "later_flag_assessment_id", "later_flag_at", "business_days_left_at_later_flag"])
missed_table["business_days_left_at_later_flag"] = missed_table["business_days_left_at_later_flag"].astype("Int64")
ever_flagged = early[flag].groupby("request_id").size()                       # any flagged scored assessment of the task
missed_table["task_ever_flagged"] = missed_table["request_id"].isin(ever_flagged.index)
missed_table["flagged_earlier"] = [bool(((early["request_id"] == r) & flag & (early["assessed_at"] < t)).any())
                                   for r, t in zip(missed_table["request_id"], missed_table["assessed_at"])]
assert (missed_table.loc[missed_table["flagged_later"], "later_flag_at"] > missed_table.loc[missed_table["flagged_later"], "assessed_at"]).all()
missed_table.insert(0, "data_version", DATA_VERSION)
missed_table.to_csv(OUTPUT_DIR / "missed_warnings_test.csv", index=False)
never = int((~missed_table.drop_duplicates("request_id")["task_ever_flagged"]).sum())       # from all scored assessments
no_later = int((~missed_table["flagged_later"]).sum())
print(f"Missed at-risk assessments: {len(missed_table)} on {missed_table['request_id'].nunique()} tasks "
      f"(by checkpoint: {missed_table['checkpoint'].value_counts().to_dict()}); "
      f"flagged at a later assessment: {int(missed_table['flagged_later'].sum())} (no later flag: {no_later}; flagged earlier: "
      f"{int(missed_table['flagged_earlier'].sum())}); tasks never warned at any scored assessment: {never}")
if len(missed_table):
    print("Business days left at the later flag:", {int(k): int(v) for k, v in missed_table["business_days_left_at_later_flag"].value_counts().sort_index().items()})
    display(missed_table.drop(columns="data_version").set_index("assessment_id"))

Missed at-risk assessments: 18 on 18 tasks (by checkpoint: {'checkpoint_1': 18}); flagged at a later assessment: 18 (no later flag: 0; flagged earlier: 0); tasks never warned at any scored assessment: 0
Business days left at the later flag: {5: 18}


,request_id,checkpoint,assessed_at,business_days_until_start,current_status,task_type,later_assessments,flagged_later,later_flag_assessment_id,later_flag_at,business_days_left_at_later_flag,task_ever_flagged,flagged_earlier
assessment_id,,,,,,,,,,,,,
ASM-00699,REQ-00568,checkpoint_1,2025-04-25 08:02:00,10,open,payroll_setup,1,True,ASM-00795,2025-05-02 08:00:00,5,True,False
ASM-01556,REQ-01271,checkpoint_1,2025-07-04 08:00:00,11,open,equipment_setup,1,True,ASM-01615,2025-07-14 08:00:00,5,True,False
ASM-01797,REQ-01463,checkpoint_1,2025-08-07 13:59:00,10,open,workplace_access,1,True,ASM-01836,2025-08-14 08:00:00,5,True,False
ASM-01802,REQ-01469,checkpoint_1,2025-08-08 09:49:00,9,open,system_access,1,True,ASM-01837,2025-08-14 08:00:00,5,True,False
ASM-01943,REQ-01582,checkpoint_1,2025-09-05 09:18:00,24,open,orientation_scheduling,1,True,ASM-02185,2025-10-03 08:00:00,5,True,False
ASM-02084,REQ-01677,checkpoint_1,2025-09-23 08:56:00,11,open,workplace_access,1,True,ASM-02176,2025-10-02 08:00:00,5,True,False
ASM-02222,REQ-01787,checkpoint_1,2025-10-09 13:54:00,10,open,payroll_setup,1,True,ASM-02274,2025-10-17 08:00:00,5,True,False
ASM-02291,REQ-01816,checkpoint_1,2025-10-17 15:49:00,10,open,equipment_setup,1,True,ASM-02340,2025-10-24 08:00:00,5,True,False
ASM-02425,REQ-01976,checkpoint_1,2025-11-06 13:17:00,10,open,equipment_setup,1,True,ASM-02466,2025-11-14 08:00:00,5,True,False


### Groups

Results by employee type, task type, intake channel, department and checkpoint, for the policy and the comparison model, computed as in notebook 04. The three labels from the frozen policy are applied **independently**, so one does not cancel another:

- **limited recall support:** fewer than 30 at-risk assessments in the group (recall rests on few cases);
- **documented concern:** recall at least 10 percentage points below the policy's own overall test recall, **or** at least one at-risk task in the group with no early-warning flag at any checkpoint — this applies whatever the group's support;
- **serious concern:** at least 30 at-risk assessments and recall below 80%.

A group with no at-risk assessments has no recall, which is reported as missing. A serious concern on test would not change the frozen policy here; it would be carried into the decision brief and model card as a human-review condition.

In [5]:
GROUPINGS = ["employee_type", "task_type", "intake_channel", "hiring_department", "checkpoint"]
GROUP_METHODS = {"rule": "days_rule", "model": "logistic_regression"}
caught_tasks = {col: set(early.loc[early[col].astype(bool) & (early["y"] == 1), "request_id"]) for col in GROUP_METHODS.values()}
overall_recall = {short: 100 * results.loc[METHODS[col], "recall"] for short, col in GROUP_METHODS.items()}
group_rows = []
for g in GROUPINGS:
    for value, rows in early.groupby(g):
        y = rows["y"] == 1
        base = {"grouping": g, "group": value, "assessments": len(rows), "at_risk": int(y.sum()), "at_risk_%": 100 * y.mean(),
                "unique_tasks": rows["request_id"].nunique(), "employees": rows["employee_id"].nunique(),
                "limited_recall_support": int(y.sum()) < GROUP_RULES["limited_support_below_at_risk"]}
        for short, col in GROUP_METHODS.items():
            p = rows[col].astype(bool)
            tp = int((p & y).sum())
            base.update({f"{short}_flagged": int(p.sum()),
                         f"{short}_recall_%": 100 * tp / int(y.sum()) if y.any() else np.nan,
                         f"{short}_precision_%": 100 * tp / int(p.sum()) if p.any() else np.nan,
                         f"{short}_flag_rate_%": 100 * p.mean(),
                         f"{short}_at_risk_tasks_never_flagged_any_checkpoint":
                             int((~rows.loc[y, "request_id"].drop_duplicates().isin(caught_tasks[col])).sum())})
        group_rows.append(base)
groups = pd.DataFrame(group_rows)
for short in GROUP_METHODS:
    groups[f"{short}_recall_gap_vs_own_overall_pp"] = groups[f"{short}_recall_%"] - overall_recall[short]
groups["documented_concern"] = ((groups["rule_recall_gap_vs_own_overall_pp"] <= GROUP_RULES["documented_concern_gap_pp"])
                                | (groups["rule_at_risk_tasks_never_flagged_any_checkpoint"] > 0))
groups["serious_concern"] = ((groups["at_risk"] >= GROUP_RULES["limited_support_below_at_risk"])
                             & (groups["rule_recall_%"] < GROUP_RULES["serious_concern_recall_below_pct"]))
assert (groups.groupby("grouping")["assessments"].sum() == len(early)).all(), "Each grouping must cover every assessment once"
groups.insert(0, "data_version", DATA_VERSION)
groups.to_csv(OUTPUT_DIR / "group_results_test.csv", index=False)
concerns = groups.loc[groups["documented_concern"] | groups["serious_concern"],
                      ["grouping", "group", "assessments", "at_risk", "limited_recall_support", "rule_recall_%",
                       "rule_recall_gap_vs_own_overall_pp", "rule_at_risk_tasks_never_flagged_any_checkpoint",
                       "documented_concern", "serious_concern"]]
concerns.to_csv(OUTPUT_DIR / "group_concerns_test.csv", index=False)
supported = groups.loc[~groups["limited_recall_support"]].dropna(subset=["rule_recall_gap_vs_own_overall_pp"])
closest = supported.loc[supported["rule_recall_gap_vs_own_overall_pp"].idxmin()] if len(supported) else None
print(f"Policy (a) overall test recall {overall_recall['rule']:.1f}%. Documented concerns: {int(groups['documented_concern'].sum())}; "
      f"serious concerns: {int(groups['serious_concern'].sum())}; groups with limited recall support: {int(groups['limited_recall_support'].sum())}")
if closest is not None:
    print(f"Lowest recall among supported groups: {closest['grouping']} = {closest['group']}, recall {closest['rule_recall_%']:.1f}% "
          f"({closest['rule_recall_gap_vs_own_overall_pp']:+.1f} points; {closest['at_risk']} at-risk assessments)")
# For each serious-concern group: where each at-risk task was first flagged
for c in concerns[concerns["serious_concern"]].itertuples():
    g_rows = early[(early[c.grouping] == c.group) & (early["y"] == 1)]
    first_flag = g_rows[g_rows["days_rule"].astype(bool)].sort_values("assessed_at").groupby("request_id").first()
    print(f"{c.grouping} = {c.group}: {g_rows['request_id'].nunique()} at-risk tasks; first flagged at "
          f"{first_flag['checkpoint'].value_counts().sort_index().to_dict()}; never flagged: "
          f"{g_rows['request_id'].nunique() - len(first_flag)}")
if len(concerns):
    display(concerns.set_index(["grouping", "group"]).round(1))
for g in GROUPINGS:
    print(f"By {g.replace('_', ' ')}:")
    display(groups[groups["grouping"] == g].drop(columns=["data_version", "grouping"]).set_index("group")
            [["assessments", "at_risk", "limited_recall_support", "rule_flagged", "rule_recall_%", "rule_precision_%",
              "rule_at_risk_tasks_never_flagged_any_checkpoint", "model_recall_%", "documented_concern", "serious_concern"]].round(1))

Policy (a) overall test recall 90.6%. Documented concerns: 3; serious concerns: 1; groups with limited recall support: 5
Lowest recall among supported groups: employee_type = permanent, recall 79.6% (-11.0 points; 49 at-risk assessments)
employee_type = permanent: 30 at-risk tasks; first flagged at {'checkpoint_1': 20, 'checkpoint_2': 10}; never flagged: 0


,,assessments,at_risk,limited_recall_support,rule_recall_%,rule_recall_gap_vs_own_overall_pp,rule_at_risk_tasks_never_flagged_any_checkpoint,documented_concern,serious_concern
grouping,group,,,,,,,,
employee_type,permanent,548,49,False,79.6,-11.0,0,True,True
intake_channel,unknown,10,2,True,50.0,-40.6,0,True,False
hiring_department,Utilities,125,15,True,80.0,-10.6,0,True,False


By employee type:


,assessments,at_risk,limited_recall_support,rule_flagged,rule_recall_%,rule_precision_%,rule_at_risk_tasks_never_flagged_any_checkpoint,model_recall_%,documented_concern,serious_concern
group,,,,,,,,,,
permanent,548,49,False,166,79.6,23.5,0,75.5,True,True
seasonal,286,94,False,203,95.7,44.3,0,95.7,False,False
temporary,222,49,False,134,91.8,33.6,0,89.8,False,False


By task type:


,assessments,at_risk,limited_recall_support,rule_flagged,rule_recall_%,rule_precision_%,rule_at_risk_tasks_never_flagged_any_checkpoint,model_recall_%,documented_concern,serious_concern
group,,,,,,,,,,
equipment_setup,167,47,False,81,89.4,51.9,0,91.5,False,False
orientation_scheduling,212,41,False,92,90.2,40.2,0,87.8,False,False
payroll_setup,270,40,False,138,87.5,25.4,0,87.5,False,False
system_access,249,35,False,115,94.3,28.7,0,85.7,False,False
workplace_access,158,29,True,77,93.1,35.1,0,93.1,False,False


... (2 outputs left out)

By hiring department:


,assessments,at_risk,limited_recall_support,rule_flagged,rule_recall_%,rule_precision_%,rule_at_risk_tasks_never_flagged_any_checkpoint,model_recall_%,documented_concern,serious_concern
group,,,,,,,,,,
Administration,129,19,True,29,84.2,55.2,0,78.9,False,False
Finance,76,10,True,32,90.0,28.1,0,90.0,False,False
Parks,226,44,False,133,93.2,30.8,0,90.9,False,False
Recreation,254,74,False,149,90.5,45.0,0,93.2,False,False
Transit,246,30,False,110,96.7,26.4,0,90.0,False,False
Utilities,125,15,True,50,80.0,24.0,0,73.3,True,False


By checkpoint:


,assessments,at_risk,limited_recall_support,rule_flagged,rule_recall_%,rule_precision_%,rule_at_risk_tasks_never_flagged_any_checkpoint,model_recall_%,documented_concern,serious_concern
group,,,,,,,,,,
checkpoint_1,829,146,False,276,87.7,46.4,0,89.0,False,False
checkpoint_2,227,46,False,227,100.0,20.3,0,89.1,False,False


In [6]:
# The test record: fingerprints, thresholds, population and results. Reruns compare with the saved record.
def clean(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return None if np.isnan(value) else float(value)
    return value


record = {
    "data_version": DATA_VERSION, "policy": policy["name"], "flag_rule": policy["flag_rule"],
    "days_threshold": DAYS_THRESHOLD, "model_threshold": MODEL_THRESHOLD,
    "fingerprints_sha256": RUN_FINGERPRINTS,
    "population": {"test_employees": int(test["employee_id"].nunique()), "test_tasks_any_assessment": int(test["request_id"].nunique()),
                   "scored_assessments": len(early), "scored_at_risk": int(early["y"].sum()), "scored_tasks": int(SCORED_TASKS),
                   "scored_employees": int(SCORED_EMPLOYEES), "overdue_assessments": len(overdue), "unknown_outcome_assessments": len(unknown)},
    "results": {m: {k: clean(v) for k, v in results.loc[label].drop("data_version").items()} for m, label in METHODS.items()},
    "ranking": {r: {k: clean(v) for k, v in ranking.loc[r].items()} for r in ranking.index},
    "overdue": {k: clean(v) for k, v in overdue_flags.items()},
    "missed_warnings": {"assessments": len(missed_table), "tasks": int(missed_table["request_id"].nunique()),
                        "by_employee_type": {k: int(v) for k, v in missed_table["request_id"].map(
                            early.drop_duplicates("request_id").set_index("request_id")["employee_type"]).value_counts().sort_index().items()},
                        "flagged_later": int(missed_table["flagged_later"].sum()), "flagged_earlier": int(missed_table["flagged_earlier"].sum()),
                        "tasks_never_warned": int(never)},
    "group_concerns": {"documented": [f"{r.grouping} = {r.group}" for r in groups.itertuples() if r.documented_concern],
                       "serious": [f"{r.grouping} = {r.group}" for r in groups.itertuples() if r.serious_concern]},
}
RECORD_PATH = OUTPUT_DIR / "test_results.json"
if RECORD_PATH.exists():
    assert json.loads(RECORD_PATH.read_text()) == record, "This run's results differ from the saved test record"
    record_status = "matches the saved record"
else:
    RECORD_PATH.write_text(json.dumps(record, indent=2) + "\n")
    record_status = "saved"
print(f"Test record ({RECORD_PATH.name}): {record_status}")

Test record (test_results.json): saved


**What section B shows.**

- **The frozen policy performed on the test employees much as on validation.** Policy (a) flagged 174 of the 192 at-risk early-warning assessments (recall 90.6%, against 91.2% on validation), with precision 34.6% and 503 flags (47.6% of assessments) on 373 of the 829 scored tasks. Per 100 test assessments it flagged about 48, about 16 of them labelled at risk, and missed about 1.7 at-risk assessments.
- **Every at-risk task was warned before the start-date deadline.** All 146 at-risk test tasks were flagged at least once, each first flag before the task's 08:00 start-date deadline (checked from timestamps); the median time left at a task's first flag was 4 business days. The 18 missed assessments were all at checkpoint 1 (9 to 26 business days before the start), and every one of their tasks was flagged at its checkpoint-2 assessment with 5 business days left. A miss therefore meant a later warning, not no warning; a warning four or five business days before the start does not by itself show that there was enough time to act on it.
- **A serious group concern appeared on test: permanent employees.** Their recall was 79.6% (39 of 49 at-risk assessments, on 30 tasks), 11.0 points below the overall test recall and below 80% with at least 30 at-risk assessments. On validation the same group was the closest to the review trigger (81.5%, −9.8 points). Ten of the 18 misses were permanent employees' tasks, nine of them 10 to 12 business days before the start, just beyond the 8-day cutoff. All 30 at-risk tasks received an early-warning flag (20 first at checkpoint 1, 10 first at checkpoint 2); the ten tasks missed at checkpoint 1 were subsequently flagged at checkpoint 2. The frozen policy is not changed here. The finding triggers review under the frozen monitoring criteria and informs the safeguards proposed in the decision brief and model card; no particular intervention for this group has been tested, and 49 at-risk assessments remain a small base for any conclusion about it.
- **Two documented concerns rest on limited support:** the unknown intake channel (1 of 2 at-risk assessments flagged) and the Utilities department (12 of 15). Their recall is far below overall, but on very few at-risk cases; no at-risk task in either group went unwarned.
- **The comparison model did not do better.** It flagged 171 of 192 (recall 89.1%, below the 90% target), with 501 flags, and left one at-risk task without any early-warning flag. Its ranking measures were higher than days until start (ROC AUC 0.871 against 0.860; average precision 0.676 against 0.602), as on validation, but at the saved thresholds this did not translate into more at-risk assessments flagged.
- **Overdue:** all 41 overdue test assessments go to the urgent-review list.

These are results for synthetic employees in data version 2: the rates describe the generator's assumptions, not real HR performance, and flags and timing describe a retrospective replay, not evidence that a live review list would save time.